# 1 · Nodes

A node is a function, and its type hints are its interface. `text: str` is an input, `-> str` an output. `Annotated[str, Param(...)]` adds what a person sees in an editor.

In [ ]:
from dataclasses import dataclass
from typing import Annotated

from conductor import Choice, NodeRegistry, Param, node
from conductor.widgets import Dropdown, Slider, Switch, Textarea, TextInput

## One input, one output

The smallest node. One output is named `result`, and a node is called like its function, which is its unit test.

In [ ]:
@node("shout")
def shout(text: str) -> str:
    return text.upper()


shout("hello")

## What a person reads

A `Param` gives an input its title and its control, and titles an output too.

In [ ]:
@node("greet", title="Greeting", description="Generates a greeting", category="text")
def greet(
    name: Annotated[str, Param(title="Name", description="Who to greet", widget=TextInput())],
) -> Annotated[str, Param(title="Greeting")]:
    return f"Hello, {name}!"


greet("World")

## A dropdown

`Dropdown(choices=...)` stores the choice `id`. A default makes an input optional.

In [ ]:
STYLES = (
    Choice(id="first-last", title="First Last"),
    Choice(id="last-first", title="Last, First"),
    Choice(id="upper-first", title="LAST First"),
)


@node("format-name", title="Format Name", description="Formats a full name", category="text")
def format_name(
    first: Annotated[str, Param(title="First name", widget=TextInput())],
    last: Annotated[str, Param(title="Last name", widget=TextInput())],
    style: Annotated[str, Param(title="Style", widget=Dropdown(choices=STYLES))] = "first-last",
) -> Annotated[str, Param(title="Full name")]:
    match style:
        case "last-first":
            return f"{last}, {first}"
        case "upper-first":
            return f"{last.upper()} {first}"
        case _:
            return f"{first} {last}"


format_name(first="Ada", last="Lovelace", style="upper-first")

## Slider, switch, textarea

In [ ]:
@node("truncate", title="Truncate", description="Truncates text to a max length", category="text")
def truncate(
    text: Annotated[str, Param(title="Text", widget=Textarea(rows=3))],
    max_length: Annotated[int, Param(ge=1, le=500, widget=Slider(step=1))] = 100,
    add_ellipsis: Annotated[bool, Param(title="Add ...", widget=Switch())] = True,
) -> Annotated[str, Param(title="Truncated")]:
    if len(text) <= max_length:
        return text
    truncated = text[:max_length]
    return f"{truncated}..." if add_ellipsis else truncated


truncate(text="The quick brown fox jumps over the lazy dog", max_length=19)

## Several outputs

A frozen dataclass: each field is an output of that name, and a `Param` on it titles it.

In [ ]:
@dataclass(frozen=True)
class Lines:
    first: Annotated[str, Param(title="First line")]
    rest: Annotated[str, Param(title="Remaining")]


@node("split-text", title="Split Text", description="Splits text into first line and rest", category="text")
def split_text(text: Annotated[str, Param(title="Text", widget=Textarea(rows=4))]) -> Lines:
    first, _, rest = text.partition("\n")
    return Lines(first=first, rest=rest)


split_text("hello\nworld\nagain")

## The registry

`@node` registers nothing. A registry is the nodes a host offers: `registry.add(...)` puts plain nodes on it, and `@registry.node(...)` takes `@node`'s arguments and adds the node in one step. A node that enters takes the registry's defaults, here its category. Looping over a registry yields the definitions, and each describes itself as JSON: the record an editor renders.

In [ ]:
registry = NodeRegistry(category="text")
registry.add(shout, greet, format_name, truncate, split_text)


@registry.node("reverse")
def reverse(text: str) -> str:
    return text[::-1]


for definition in registry:
    described = definition.describe()
    inputs = ", ".join(f"{field['name']}: {field['type']}" for field in described["inputs"])
    outputs = ", ".join(f"{field['name']}: {field['type']}" for field in described["outputs"])
    print(f"  {definition.id}: {definition.title} ({definition.category})")
    print(f"    inputs:  ({inputs})")
    print(f"    outputs: ({outputs})")
    print()

## Registries from modules

A module can keep a registry of its own, with defaults for its nodes, and the host adds it at the top with `add_subregistry`: its nodes are copied in, its tags first. A second node under one id raises. `filter` gives a narrower registry, say the nodes one customer may place; the predicate gets the whole definition.

In [ ]:
beta = NodeRegistry(tags=["beta"])


@beta.node("whisper")
def whisper(text: str) -> str:
    return text.lower()


registry.add_subregistry(beta)
stable = registry.filter(lambda definition: "beta" not in definition.tags)
[definition.id for definition in registry], [definition.id for definition in stable]